# brain_spi — FBIRN quickstart

End-to-end walkthrough of the library on the FBIRN ICA dataset (311 subjects, 140 timepoints, 53 ICA components).

**Sections**
1. Load data
2. Inspect dataset
3. Quick sanity check — Pearson CC baseline
4. Run the full SPI pipeline
5. Inspect per-SPI results
6. Cross-SPI aggregate
7. Robustness: bootstrap resampling
8. True null: label-shuffle
9. Save / reload result

In [ ]:
import sys, os
# make sure the repo root is on the path
sys.path.insert(0, os.path.join(os.path.dirname(''), '..'))

import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt

from src.load_data import load_data
import brain_spi
from brain_spi import BrainSPI, PipelineResult
import brain_spi.domains as domains

print('brain_spi loaded OK')

## 1. Load data

In [ ]:
fbirn, demo_df = load_data()

timeseries = fbirn['data']    # (311, 140, 53)
labels     = fbirn['diags']   # 0=HC, 1=SZ
sexes      = fbirn['sexes']
ages       = fbirn['ages']

print('Data shape  :', timeseries.shape)
print('Labels shape:', labels.shape)
print('HC / SZ     :', (labels==0).sum(), '/', (labels==1).sum())
display(demo_df)

## 2. Domain / network labels (Neuromark ICA)

The `ICN_coordinates.csv` file assigns each of the 53 components to a named network. We build a `DomainSpec` that the plotting helpers use to draw grid guides.

In [ ]:
domain_spec = domains.from_csv('../data/ICN_coordinates.csv', name_col='Domain')

print('Domains:', domain_spec.names)
print('ROI count:', domain_spec.n_rois)

## 3. Sanity check — Pearson correlation baseline

This reproduces the notebook's PCC baseline using `brain_spi._utils.corrcoef_batch`, then plots group means and t-test results.

In [ ]:
from brain_spi._utils import corrcoef_batch
from brain_spi.stats import ttest_edges, rf_features
from brain_spi.viz import plot_heatmap, plot_triptych
from brain_spi.pipeline import PerSPIResult

pcc = corrcoef_batch(timeseries)   # (311, 53, 53)
print('PCC matrices shape:', pcc.shape)

t_stat, p_value, p_thresh = ttest_edges(pcc, labels)
density = p_thresh.astype(float)[np.tril_indices(53, k=-1)].mean()
rf_imp, rf_mask = rf_features(pcc, labels, density=density,
                               rf_kw={'n_estimators': 500, 'random_state': 0})

pcc_result = PerSPIResult(
    name='PCC',
    matrices=pcc,
    labels=labels,
    group_values=np.array([0, 1]),
    t_stat=t_stat,
    p_value=p_value,
    p_thresh=p_thresh,
    rf_importance=rf_imp,
    rf_mask=rf_mask,
)

print(f'Bonferroni-sig edges: {p_thresh.sum()//2} / {53*52//2}')
print(f'AND edges            : {pcc_result.and_mask.sum()//2}')

fig = pcc_result.plot_triptych(domain_spec=domain_spec)
fig.suptitle('PCC baseline', y=1.01)
plt.show()

## 4. Full SPI pipeline

Run `BrainSPI.fit()` with the default 9-SPI set. The first run computes pyspi calculators (takes a few minutes); results are cached to `~/.cache/brain_spi/` so subsequent runs are near-instant.

**To run quickly on a subset first**, uncomment the `timeseries[:20]` lines below.

In [ ]:
# Optional: smoke-test on a small subset first
# _data   = timeseries[:20]
# _labels = labels[:20]

_data   = timeseries
_labels = labels

pipe = BrainSPI(
    spis=None,           # default 9-SPI set
    # spis='spis_all',     # full pyspi config
    # spis=['kendalltau', 'spearmanr'],  # explicit subset
    group_names=('HC', 'SZ'),
)

result = pipe.fit(_data, _labels)
print('SPIs computed:', result.spis)

In [ ]:
result

In [ ]:
result()

## 5. Per-SPI inspection

In [ ]:
# Pick one SPI to inspect in detail
spi_name = 'kendalltau'

r = result[spi_name]
print(f'{spi_name} — matrices shape: {r.matrices.shape}')
print(f'Sig-p edges (Bonferroni): {r.p_thresh.sum()//2}')
print(f'RF mask edges           : {r.rf_mask.sum()//2}')
print(f'AND mask edges          : {r.and_mask.sum()//2}')

fig = r.plot_triptych(domain_spec=domain_spec)
plt.show()

In [ ]:
# Group mean connectivity
fig = r.plot_mean(by_group=True, group_names=('HC', 'SZ'), domain_spec=domain_spec)
plt.show()

In [ ]:
# All SPIs at once — poster-style grid
from brain_spi.viz import plot_spi_grid

fig = plot_spi_grid(
    {name: result[name] for name in result.spis},
    domain_spec=domain_spec,
    # save='../pics/spi_grid.png',
)
plt.show()

## 6. Cross-SPI aggregate

`result.aggregate` is computed lazily on first access. `mean_and[i,j]` is the fraction of SPIs that flagged edge (i,j) in their AND mask.

In [ ]:
agg = result.aggregate

print('mean_and shape:', agg.mean_and.shape)
print('mean_and range:', agg.mean_and.min().round(3), '—', agg.mean_and.max().round(3))
print('Edges flagged by ≥1 SPI  :', (agg.mean_and > 0).sum() // 2)
print('Edges flagged by all SPIs:', (agg.mean_and == 1).sum() // 2)

fig = agg.plot(domain_spec=domain_spec)
plt.show()

In [ ]:
# Side-by-side: mean_p_thresh vs mean_rf_mask vs mean_and
from brain_spi.viz import plot_heatmap

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for mat, title, ax in [
    (agg.mean_p_thresh, 'Mean sig-p (Bonferroni)', axes[0]),
    (agg.mean_rf_mask,  'Mean RF mask',            axes[1]),
    (agg.mean_and,      'Mean AND',                axes[2]),
]:
    im = plot_heatmap(mat, ax=ax, cmap='inferno', vmin=0, vmax=1,
                      guides=domain_spec, title=title)
    fig.colorbar(im, ax=ax, fraction=0.045)
fig.tight_layout()
plt.show()

## 7. Robustness: bootstrap subject resampling

Draws `n` random 2/3-subject subsets, reruns stages 2–3, and returns a distribution of `mean_and` matrices. High per-edge survival rate → the finding isn't driven by a few subjects.

In [ ]:
boot = result.bootstrap(n=20, frac=0.66)

print('Bootstrap samples shape:', boot.samples.shape)  # (20, 53, 53)
print('Mean across iterations  :', boot.mean.mean().round(4))
print('Std across iterations   :', boot.std.mean().round(4))

In [ ]:
# Survival rate: fraction of bootstrap iterations that replicate each edge
survival = boot.survival_rate(agg.mean_and)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = plot_heatmap(boot.mean, ax=axes[0], cmap='inferno', vmin=0, vmax=1,
                   guides=domain_spec, title='Bootstrap mean')
fig.colorbar(im0, ax=axes[0], fraction=0.045)

im1 = plot_heatmap(survival, ax=axes[1], cmap='plasma', vmin=0, vmax=1,
                   guides=domain_spec, title='Survival rate (≥ observed)')
fig.colorbar(im1, ax=axes[1], fraction=0.045)
fig.tight_layout()
plt.show()

## 8. True null: label-shuffle permutation test

Shuffles group labels `n` times. The resulting distribution should be centred near zero — significantly above it means the observed `mean_and` isn't a chance result.

In [ ]:
null = result.label_shuffle(n=100)

print('Null samples shape:', null.samples.shape)   # (100, 53, 53)
print('Null mean (global):', null.mean.mean().round(4))
print('Null std  (global):', null.std.mean().round(4))

In [ ]:
# Permutation p-value per edge
perm_p = null.survival_rate(agg.mean_and)   # fraction of null >= observed

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = plot_heatmap(null.mean, ax=axes[0], cmap='inferno', vmin=0, vmax=1,
                   guides=domain_spec, title='Label-shuffle null mean')
fig.colorbar(im0, ax=axes[0], fraction=0.045)

im1 = plot_heatmap(perm_p, ax=axes[1], cmap='magma_r', vmin=0, vmax=0.1,
                   guides=domain_spec, title='Permutation p-value')
fig.colorbar(im1, ax=axes[1], fraction=0.045)
fig.tight_layout()
plt.show()

print(f'Edges with perm-p < 0.05: {(perm_p < 0.05).sum()//2}')

## 9. Save and reload

In [ ]:
out_path = '/tmp/brain_spi_fbirn_result.pkl'
result.to_pickle(out_path)
print('Saved to', out_path)

result2 = PipelineResult.load_pickle(out_path)
print('Reloaded SPIs:', result2.spis)

# verify round-trip
for name in result.spis:
    np.testing.assert_array_equal(result[name].matrices, result2[name].matrices)
print('All matrices identical — round-trip OK')